Import the libraries.

In [13]:
from sklearn.datasets import load_diabetes

import numpy as np
import random
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

Load the premade dataset, train the model with linear regression and print the r2 score.

In [14]:
X,y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=2)
model = LinearRegression()
model.fit(X_train, y_train)
print(model.coef_, model.intercept_)
y_pred = model.predict(X_test)
print(r2_score(y_test, y_pred))

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238] 151.88331005254167
0.4399338661568968


Self made Mini-batch GD function.

In [ ]:
class SelfMadeGD:
    def __init__(self, epochs, learning_rate, batch_size):
        self.intercept_ = None
        self.coef_ = None
        self.epochs = epochs
        self.lr = learning_rate
        self.batch_size = batch_size

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.intercept_ = 0
        self.coef_ = np.zeros(n_features)

        for epoch in range(self.epochs):
            # shuffle once per epoch
            indices = np.random.permutation(n_samples) # shuffle indices
            X_shuffled = X[indices] # shuffle X using shuffled indices
            y_shuffled = y[indices] # shuffle y using shuffled indices

            # batch loop
            for i in range(0, n_samples, self.batch_size): # loop through batches
                X_batch = X_shuffled[i:i+self.batch_size] # get batch of X
                y_batch = y_shuffled[i:i+self.batch_size] # get batch of y

                y_hat = self.intercept_ + np.dot(X_batch, self.coef_)
                error = y_batch - y_hat

                intercept_derivative = -2 * np.mean(error)
                coef_derivative = -2 * np.dot(X_batch.T, error) / len(X_batch)

                self.intercept_ -= self.lr * intercept_derivative
                self.coef_ -= self.lr * coef_derivative

    def predict(self, X):
        return self.intercept_ + np.dot(X, self.coef_)

Train the model and print r2 score.

In [16]:
mbr = SelfMadeGD(learning_rate=0.01, epochs=500, batch_size=int(len(X_train)/50))
mbr.fit(X_train, y_train)
print(mbr.coef_, mbr.intercept_)
print(r2_score(y_test, mbr.predict(X_test)))

[  43.55887066 -105.49410023  407.93741591  281.01923801   -4.34525382
  -63.01956613 -186.82292895  122.72845709  368.18593521  124.26480228] 153.0901791228063
0.4476011057255058


Train the model with sklearn's SGDregressor class and print the r2 score.

In [ ]:
from sklearn.linear_model import SGDRegressor
sgd = SGDRegressor(max_iter=500, learning_rate='constant', eta0=0.01, random_state=2)
batch_size = 35

for i in range(100):
    indices = np.random.permutation(len(X_train)) # shuffle indices

    for i in range(0, len(X_train), batch_size): # loop through batches
        batch_idx = indices[i:i+batch_size] # get batch indices
        sgd.partial_fit(X_train[batch_idx], y_train[batch_idx]) # fit batch to model 
        # partial fit is used for online learning and mini batch gradient descent it allows us to fit the model on a batch of data and update the model parameters after each batch.
        

print(sgd.coef_, sgd.intercept_)
print(r2_score(y_test, sgd.predict(X_test)))

[  55.67534813  -66.61478088  347.48329168  245.82033147   18.25342314
  -27.34608601 -172.57265815  128.923449    316.67714159  128.48318473] [151.00169119]
0.43282549296647044
